# EfficientNet-Lite0

This standalone notebook trains **EfficientNet-Lite0**, evaluates the float model, exports dynamic-range and full INT8 TFLite models, benchmarks CPU inference, and archives outputs.

## Download the Kaggle dataset into Colab

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
%pip install -q kaggle

!kaggle datasets download -d mdhasibultamim/multi-crop-leaf-disease -p /content

import os
import zipfile

KAGGLE_DATASET_BASE = "/content/dataset"
os.makedirs(KAGGLE_DATASET_BASE, exist_ok=True)
dataset_archive = "/content/multi-crop-leaf-disease.zip"
with zipfile.ZipFile(dataset_archive) as archive:
    archive.extractall(KAGGLE_DATASET_BASE)
print("Extracted Kaggle dataset to:", KAGGLE_DATASET_BASE)

In [ ]:
import os

SOURCE_DATASET_BASE = os.path.join(KAGGLE_DATASET_BASE, "processed")
if not os.path.isdir(SOURCE_DATASET_BASE):
    raise FileNotFoundError(f"Downloaded source dataset not found: {SOURCE_DATASET_BASE}")
print("Raw Kaggle source dataset:", SOURCE_DATASET_BASE)


## Clone Project Repository

In [ ]:
from google.colab import drive
import os
import subprocess

drive.mount("/content/drive")

ARCH = "efficientnet_lite0"
REPO_URL = "https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git"
REPO_DIR = "/content/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System"
OUTPUT_BASE = f"/content/drive/MyDrive/leaf_models/{ARCH}"
print("Configured:", ARCH)
print("Outputs:", OUTPUT_BASE)


if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    sync_result = subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main"], text=True, capture_output=True)
    if sync_result.returncode == 0:
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
    print(sync_result.stdout, sync_result.stderr)
required_paths = ["requirements.txt", "training/train.py", "training/evaluate.py", "quantization/post_training_quant.py"]
missing_paths = [path for path in required_paths if not os.path.isfile(os.path.join(REPO_DIR, path))]
if missing_paths:
    raise RuntimeError(f"Repository clone is incomplete; missing: {missing_paths}")
print("Complete project repository is ready:", REPO_DIR)

## Install project dependencies

In [ ]:
import os
import subprocess
import sys

requirements_path = os.path.join(REPO_DIR, "requirements.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path], check=True)
print("Dependencies installed from:", requirements_path)

## Configure paths and detect classes

In [ ]:
import csv
import json
import os
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, REPO_DIR)
from dataset.taxonomy import CANONICAL_CLASSES

SOURCE_DATASET_BASE = os.path.join(KAGGLE_DATASET_BASE, "processed")
DATASET_BASE = "/content/drive/MyDrive/leaf_data/processed_harmonized"
class_names = list(CANONICAL_CLASSES)


def validate_clean_benchmark(dataset_dir):
    dataset_dir = Path(dataset_dir)
    class_manifest = dataset_dir / "class_names.json"
    split_manifest = dataset_dir / "split_manifest.csv"
    if not class_manifest.is_file() or not split_manifest.is_file():
        raise ValueError(
            f"Existing benchmark is missing class_names.json or split_manifest.csv: {dataset_dir}"
        )

    try:
        names = json.loads(class_manifest.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as exc:
        raise ValueError(f"Could not read class manifest: {class_manifest}") from exc
    if names != list(CANONICAL_CLASSES):
        raise ValueError(
            f"Invalid canonical class manifest at {class_manifest}; "
            "expected the repository's ordered 72-class taxonomy."
        )

    expected = set(CANONICAL_CLASSES)
    for split in ("train", "val", "test"):
        split_dir = dataset_dir / split
        if not split_dir.is_dir():
            raise FileNotFoundError(f"Benchmark split is missing: {split_dir}")
        actual = {
            child.name for child in split_dir.iterdir()
            if child.is_dir()
        }
        if actual != expected:
            raise ValueError(
                f"{split} folders in existing benchmark are invalid; "
                f"missing={sorted(expected - actual)}, extra={sorted(actual - expected)}"
            )

    seen_hashes = {}
    splits_seen = set()
    train_classes = set()
    row_count = 0
    with split_manifest.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        required = {"split", "canonical_label", "source_sha256", "output_path"}
        if not reader.fieldnames or not required.issubset(reader.fieldnames):
            raise ValueError(f"Invalid split manifest columns: {split_manifest}")
        for row in reader:
            split = row["split"]
            label = row["canonical_label"]
            digest = row["source_sha256"]
            output_path = Path(row["output_path"])
            if split not in {"train", "val", "test"} or label not in expected or not digest:
                raise ValueError(f"Invalid split manifest row in {split_manifest}: {row}")
            if len(output_path.parts) < 3 or output_path.parts[:2] != (split, label):
                raise ValueError(f"Invalid output path in split manifest: {row['output_path']!r}")
            if not (dataset_dir / output_path).is_file():
                raise FileNotFoundError(
                    f"Split manifest points to a missing image: {dataset_dir / output_path}"
                )
            previous_split = seen_hashes.get(digest)
            if previous_split is not None:
                raise ValueError(
                    f"Existing benchmark contains repeated source image hashes "
                    f"({previous_split} and {split}); refusing to reuse {dataset_dir}"
                )
            seen_hashes[digest] = split
            splits_seen.add(split)
            if split == "train":
                train_classes.add(label)
            row_count += 1

    if row_count == 0 or splits_seen != {"train", "val", "test"}:
        raise ValueError(f"Split manifest is empty or missing a split: {split_manifest}")
    if train_classes != expected:
        raise ValueError(
            f"Training manifest does not cover all canonical classes; "
            f"missing={sorted(expected - train_classes)}"
        )


if os.path.exists(DATASET_BASE):
    validate_clean_benchmark(DATASET_BASE)
    print(f"Reusing validated deduplicated benchmark: {DATASET_BASE}")
else:
    if not os.path.isdir(SOURCE_DATASET_BASE):
        raise FileNotFoundError(f"Downloaded source dataset not found: {SOURCE_DATASET_BASE}")
    os.makedirs(os.path.dirname(DATASET_BASE), exist_ok=True)
    preparation_script = os.path.join(REPO_DIR, "dataset", "prepare_benchmark.py")
    subprocess.run(
        [
            sys.executable,
            preparation_script,
            "--data-dir", SOURCE_DATASET_BASE,
            "--output-dir", DATASET_BASE,
            "--seed", "42",
        ],
        check=True,
    )
    validate_clean_benchmark(DATASET_BASE)
    print(f"Prepared deduplicated benchmark: {DATASET_BASE}")

print(f"Canonical classes: {len(class_names)}")
import yaml

cfg_path = os.path.join(REPO_DIR, "training", "config_efficientnet_lite0.yaml")
with open(cfg_path, "r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)
cfg["model"]["hub_url"] = "https://tfhub.dev/tensorflow/efficientnet/lite0/feature-vector/2"

cfg["dataset"]["data_dir"] = DATASET_BASE
for split in ("train", "val", "test"):
    cfg["dataset"][f"{split}_dir"] = os.path.join(DATASET_BASE, split)
cfg["model"]["num_classes"] = len(class_names)
# Keep the full dataset off RAM on hosted runtimes.
cfg["dataset"]["cache_mode"] = "disk"
cfg["dataset"]["cache_memory_limit_gb"] = 8.0
cfg.setdefault("performance", {}).update({
    "mixed_precision": True,
    "steps_per_execution": 32,
    "jit_compile": False,
    "deterministic": False,
    "distribution": "auto",
    "log_throughput": True,
})
cfg["export"]["save_dir"] = os.path.dirname(OUTPUT_BASE)
cfg["callbacks"]["csv_logger"]["filename"] = os.path.join(OUTPUT_BASE, f"training_log_{ARCH}.csv")
cfg["callbacks"]["tensorboard"]["log_dir"] = os.path.join(OUTPUT_BASE, "tensorboard")
os.makedirs(OUTPUT_BASE, exist_ok=True)
with open(cfg_path, "w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)
print(f"{ARCH}: {len(class_names)} classes")
print("Config:", cfg_path)

# Build-only preflight: verify TensorFlow Hub and Lite0 before starting a long run.
preflight_code = f"""
import os, sys
os.environ.setdefault('TF_USE_LEGACY_KERAS', '1')
sys.path.insert(0, {REPO_DIR!r})
from training.model import get_model
model, _ = get_model(
    'efficientnet_lite0',
    input_shape=(224, 224, 3),
    num_classes={len(class_names)},
    dropout_rate={cfg['model'].get('dropout_rate', 0.5)!r},
    weights='imagenet',
    hub_url={cfg['model'].get('hub_url')!r},
    hub_cache_dir={os.environ['TFHUB_CACHE_DIR']!r},
)
print('Lite0 preflight OK:', model.output_shape)
"""
preflight_env = os.environ.copy()
preflight_env['PYTHONUNBUFFERED'] = '1'
preflight_result = subprocess.run([sys.executable, '-c', preflight_code], text=True, capture_output=True, env=preflight_env)
print(preflight_result.stdout)
if preflight_result.stderr:
    print(preflight_result.stderr)
if preflight_result.returncode != 0:
    raise RuntimeError('EfficientNet-Lite0 preflight failed. Fix the printed TensorFlow/TF Hub error before training.')

## Train the model

In [ ]:
train_env = os.environ.copy()
train_env["PYTHONUNBUFFERED"] = "1"
train_result = subprocess.run([
    sys.executable,
    os.path.join(REPO_DIR, "training", "train.py"),
    "--config",
    cfg_path,
], env=train_env)
if train_result.returncode != 0:
    raise RuntimeError(f"EfficientNet-Lite0 training failed with exit code {train_result.returncode}. Review the train.py error printed above.")

## Evaluate the float model

In [ ]:
from pathlib import Path

arch_dir = Path(OUTPUT_BASE)
saved_models = sorted(arch_dir.glob("saved_model_*"), key=lambda path: path.stat().st_mtime)
h5_models = sorted(arch_dir.glob("*.h5"), key=lambda path: path.stat().st_mtime)
if saved_models:
    model_path = str(saved_models[-1])
elif h5_models:
    model_path = str(h5_models[-1])
else:
    raise FileNotFoundError(f"No exported model found in {OUTPUT_BASE}")

float_report = os.path.join(OUTPUT_BASE, "evaluation_float")
os.makedirs(float_report, exist_ok=True)
subprocess.run([
    sys.executable,
    os.path.join(REPO_DIR, "training", "evaluate.py"),
    "--model", model_path,
    "--config", cfg_path,
    "--results-dir", float_report,
], check=True)
print("Evaluated:", model_path)

## Export dynamic-range and full INT8 TFLite models

In [ ]:
quant_script = os.path.join(REPO_DIR, "quantization", "post_training_quant.py")
quant_dir = os.path.join(OUTPUT_BASE, "quantized")
os.makedirs(quant_dir, exist_ok=True)

dynamic_path = os.path.join(quant_dir, f"{ARCH}_dynamic.tflite")
subprocess.run([
    sys.executable, quant_script,
    "--model_path", model_path,
    "--output_path", dynamic_path,
    "--arch", ARCH,
], check=True)

int8_path = os.path.join(quant_dir, f"{ARCH}_int8.tflite")
subprocess.run([
    sys.executable, quant_script,
    "--model_path", model_path,
    "--output_path", int8_path,
    "--representative_data", cfg["dataset"]["train_dir"],
    "--evaluate",
    "--test_data", cfg["dataset"]["test_dir"],
    "--arch", ARCH,
], check=True)
print("Exported:", dynamic_path, int8_path)

## Benchmark and evaluate TFLite models

In [ ]:
import json
import time
import numpy as np
import tensorflow as tf

benchmark_rows = []
for tflite_path in (dynamic_path, int8_path):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    input_shape = input_details["shape"]
    if input_details["dtype"] == np.uint8:
        dummy = np.random.randint(0, 256, size=input_shape, dtype=np.uint8)
    else:
        dummy = np.random.uniform(-1.0, 1.0, size=input_shape).astype(np.float32)
    for _ in range(10):
        interpreter.set_tensor(input_details["index"], dummy)
        interpreter.invoke()
    times = []
    for _ in range(50):
        start = time.perf_counter()
        interpreter.set_tensor(input_details["index"], dummy)
        interpreter.invoke()
        times.append((time.perf_counter() - start) * 1000.0)
    row = {
        "model": os.path.basename(tflite_path),
        "input_dtype": np.dtype(input_details["dtype"]).name,
        "mean_ms": float(np.mean(times)),
        "std_ms": float(np.std(times)),
        "size_mb": os.path.getsize(tflite_path) / (1024 * 1024),
    }
    benchmark_rows.append(row)
    print(json.dumps(row, indent=2))

with open(os.path.join(OUTPUT_BASE, "tflite_benchmark.json"), "w", encoding="utf-8") as handle:
    json.dump(benchmark_rows, handle, indent=2)

## Archive outputs

In [ ]:
import shutil

archive_base = os.path.join(os.path.dirname(OUTPUT_BASE), f"{ARCH}_outputs")
archive_path = shutil.make_archive(archive_base, "zip", OUTPUT_BASE)
print("Archive:", archive_path)

## Outputs

The model, class labels, logs, float evaluation report, quantized TFLite files, benchmark JSON, and ZIP archive are stored under the configured output directory.